# Graph-structural baseline (2-hop heterogeneous mean-aggregation GNN)

Completes the baseline ladder (RBAC -> ABAC -> ML -> DL -> RL -> **graph-structural** -> TrustMesh, per `benchmark_labels.py`'s own docstring list). Every earlier baseline scores ONE session row using only that row's own features. This one additionally lets a session's score depend on an aggregate of every OTHER session sharing its CPO, its eMSP, or its EV (hop 1), and on what THOSE entities' own neighborhoods look like one level further out (hop 2) -- real multi-hop message passing over the session-entity graph, implemented in plain PyTorch via differentiable scatter-mean (no `torch_geometric` dependency, so nothing here depends on that install succeeding in a given Colab runtime).

**Why entities carry no learned identity embedding:** `ml_features.py`'s own BUG LOG documents why a model that can see WHICH name is malicious collapses to a lookup table (AUROC/AUPRC=1.0000 exactly) instead of a behavioral detector. A learned per-entity embedding table (keyed by `cpo_id`/`emsp_id`/`ev_id`) is exactly that lookup table with extra steps, and would also silently break the zero-shot cross-variant protocol. Every entity representation here is instead a deterministic, non-learned aggregate of neighboring sessions' behavioral features -- see `graph_baseline.py`'s module docstring for the full account, including its causality rules (which rows are allowed to contribute to which entity's aggregate, and when) and one disclosed caveat about evaluation-time context, same disclosure discipline as `temporal_split.py`'s own caveat.

**Same features, same protocol as `05`/`07`/`08`:** `ml_features.ALL_FEATURES`, train-once-on-`main`/zero-shot-eval-on-every-variant, the same world_id-keyed predictions-CSV convention. **Full-batch training** (not mini-batched) -- the module docstring explains why this is the standard, expected way to train a graph model at this scale, not a shortcut.

**GPU recommended**, same reason as `07`/`08`: this trains by gradient descent, now over scatter-mean graph operations as well as dense layers.

**Requires:** `v2g_benchmark_multi_world.csv` from `02_generate_v2g_benchmark.ipynb` already in your Drive.


## Step 1: check for a GPU

If this prints "No GPU detected," switch the runtime now (Runtime -> Change runtime type -> a GPU, e.g. T4) before continuing -- changing it later would mean re-running every cell from the top anyway, since nothing has been written yet at this point.


In [ ]:
import torch

print(f"PyTorch {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected -- training will run on CPU (slower, but still correct). "
          "Runtime > Change runtime type > GPU (e.g. T4) if you want it faster.")


## Step 2: write the shared modules (decision layer, labels, split, features)

Identical to `05`/`07`/`08`'s own module-writing step -- `decision_engine.py`, `benchmark_labels.py`, `temporal_split.py`, `ml_features.py` are reused byte-for-byte, so this baseline is judged on exactly the same data/features/target as every other one in the ladder.


In [ ]:
%%writefile decision_engine.py
"""
decision_engine.py
-------------------
Implements Algorithm 4 from the TrustMesh framework doc (Section 8):
the trust-weighted three-way access decision (PERMIT / CONDITIONAL-PERMIT / DENY).

This is what turns a trust score into an actual access-control outcome,
which is the concrete instantiation of Gap 4.3.1 (fuzzy trust feeding
MA-ABAC decisions directly, not sitting beside them).
"""

from enum import Enum


class Decision(Enum):
    PERMIT = "PERMIT"
    CONDITIONAL_PERMIT = "CONDITIONAL-PERMIT"
    DENY = "DENY"


def trust_weighted_decision(base_confidence: float, effective_trust: float,
                             theta_high: float = 0.75, theta_low: float = 0.4) -> Decision:
    """
    effective_confidence = base_confidence * effective_trust   (framework doc Section 8)

    base_confidence: how strongly the attribute itself is asserted (e.g., a
    validly-signed certificate = 1.0; a partially-expired one might be lower).
    effective_trust: E[opinion] = b + a*u for the issuing authority.
    """
    effective_confidence = base_confidence * effective_trust
    if effective_confidence >= theta_high:
        return Decision.PERMIT
    elif effective_confidence >= theta_low:
        return Decision.CONDITIONAL_PERMIT
    else:
        return Decision.DENY


# ---------------------------------------------------------------------------
# Multi-attribute ABAC policy (combines authority_trust with OTHER
# attributes via rules, not just a single trust threshold -- this is what
# makes it attribute-based access control rather than trust-gating alone).
# ---------------------------------------------------------------------------

ROAMING_TRUST_PENALTY = 0.10  # roaming sessions apply a direct penalty to the
                                # CPO's trust value -- a real, common pattern
                                # (foreign/roaming access needs stronger
                                # assurance than home-network access)


def evaluate_access_policy(cpo_trust: float, emsp_trust: float, certificate_valid: bool,
                             is_roaming: bool, is_peak_hour: bool,
                             theta_high: float = 0.75, theta_low: float = 0.4,
                             roaming_penalty: float = ROAMING_TRUST_PENALTY) -> Decision:
    """
    GENUINE multi-authority ABAC policy: the decision is a function of TWO
    independently-computed authorities' trust scores, not one authority's
    trust plus trust-blind attribute checks.

      - cpo_trust:  TrustMesh's trust in the CPO (billing honesty)
      - emsp_trust: the hub-mediated, EFFECTIVE confidence in the eMSP
                    (certificate-issuance track record), already discounted
                    by how much the relying CPO trusts the hub's reporting
                    (see trust_pipeline.effective_emsp_confidence) -- this is
                    the actual cross-domain fusion mechanism validated in
                    the cold-start experiment, not a flat heuristic.
      - certificate_valid: a per-session, OBJECTIVE fact (hard constraint,
                    e.g. a CRL check) -- NOT subject to either authority's
                    trust score, since it is independently verifiable.

    Combination rule: MIN (weakest-link / AND-conjunction) of the two
    authority trust scores -- both the charging network's billing honesty
    AND the driver's home-network credential issuer must independently
    clear the bar; one being excellent cannot compensate for the other
    being poor. This is what "multi-authority" means operationally: neither
    score alone determines the outcome.
    """
    if not certificate_valid:
        return Decision.DENY  # objective fact, not subject to either authority's trust

    adjusted_cpo_trust = cpo_trust - (roaming_penalty if is_roaming else 0.0)
    combined_trust = min(adjusted_cpo_trust, emsp_trust)

    decision = trust_weighted_decision(1.0, combined_trust, theta_high, theta_low)
    if is_peak_hour and decision == Decision.PERMIT:
        decision = Decision.CONDITIONAL_PERMIT  # environment override, downgrade only

    return decision


In [ ]:
%%writefile benchmark_labels.py
"""
benchmark_labels.py
----------------------
Ground-truth labels shared by EVERY baseline (RBAC, ABAC, ML, DL, RL,
graph-structural) and by TrustMesh itself, so all of them are scored
against the exact same target.

GROUNDING LEDGER:
  - gt_cpo_malicious: directly from injection.py's authority_malicious
    column (1 iff cpo_id == the injected malicious CPO AND day_idx >=
    onset_day). Already a column in v2g_benchmark_multi_world.csv --
    not recomputed here, just carried through.
  - gt_emsp_malicious: ONSET-GATING FIX -- this USED to be a pure, permanent
    function of emsp_id == MALICIOUS_EMSP (identity_assignment.py's eMSP-C),
    with zero temporal variation: the transformer-baseline entity-attention
    proof of concept's own entity-only ablation confirmed this made the axis
    structurally unlearnable from behavior for ANY entity-aggregation model
    (mean- or attention-pooling), since there was never a honest-vs-malicious
    contrast in eMSP-C's own history to learn from, only its name. Now reads
    multi_world_pipeline.py/feature_engineering.py's persisted
    emsp_malicious_eligible column directly (1 iff emsp_id == MALICIOUS_EMSP
    AND day_idx >= emsp_onset_day) -- the same "identity AND onset" shape
    authority_malicious already has for CPO, so the two axes are symmetric.
    A combined CSV that predates this fix won't have that column;
    add_ground_truth_labels raises a clear error rather than silently
    falling back to the old identity-only definition. Independent threat
    axis from CPO billing fraud (independent, staggered onset day).
  - gt_ev_sybil: directly from enriched_features.py's ev_sybil column.
    Independent threat axis from both of the above.
  - risk_factor_count / composite_risk_score: NOT data fields -- these
    are EVALUATION-TARGET design choices made HERE, not anything the
    generation pipeline produces or implies. Said plainly so they can
    be cited precisely in the paper rather than mistaken for something
    collected "as-is": risk_factor_count is a simple count (0-3) of how
    many of the three independent threat axes are active for a given
    session; composite_risk_score is that count turned into [0,1] under
    an EQUAL-WEIGHTING assumption across axes (AXIS_WEIGHTS below) --
    not a claim that CPO billing fraud, eMSP credential issues, and EV
    Sybil attacks carry equal real-world severity.

WHY NOT A BAKED-IN "DECISION" COLUMN HERE: every baseline's own
continuous score gets passed through decision_engine.trust_weighted_decision()
at evaluation time (04_performance_analysis.ipynb), not pre-computed here.
That keeps the PERMIT / CONDITIONAL-PERMIT / DENY granularity a property
of each model's score, not of the ground truth -- which is what lets the
benchmark show every baseline (not just TrustMesh) supporting that same
granularity, and lets a user push to a finer granularity later just by
changing threshold cut points, with nothing in this module to revisit.
"""

import numpy as np
import pandas as pd

MALICIOUS_EMSP = "eMSP-C"  # mirrors identity_assignment.py; duplicated here
                            # deliberately so this module has no import-time
                            # dependency on trustmesh_pkg's package layout

AXIS_WEIGHTS = {"cpo": 1.0, "emsp": 1.0, "ev": 1.0}  # documented equal-weighting
                                                        # assumption, see module docstring


def add_ground_truth_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["gt_cpo_malicious"] = df["authority_malicious"].astype(int)

    if "emsp_malicious_eligible" not in df.columns:
        raise KeyError(
            "Column 'emsp_malicious_eligible' not found. This combined CSV predates "
            "the eMSP onset-gating fix (see multi_world_pipeline.py / "
            "feature_engineering.py, and this module's GROUNDING LEDGER above) -- "
            "regenerate the dataset (02_generate_v2g_benchmark.ipynb) before running "
            "this baseline. Refusing to silently fall back to the old, structurally-"
            "unlearnable identity-only definition (emsp_id == MALICIOUS_EMSP)."
        )
    df["gt_emsp_malicious"] = df["emsp_malicious_eligible"].astype(int)
    df["gt_ev_sybil"] = df["ev_sybil"].astype(int)

    df["risk_factor_count"] = (
        df["gt_cpo_malicious"] + df["gt_emsp_malicious"] + df["gt_ev_sybil"]
    )
    total_weight = sum(AXIS_WEIGHTS.values())
    df["composite_risk_score"] = (
        AXIS_WEIGHTS["cpo"] * df["gt_cpo_malicious"]
        + AXIS_WEIGHTS["emsp"] * df["gt_emsp_malicious"]
        + AXIS_WEIGHTS["ev"] * df["gt_ev_sybil"]
    ) / total_weight
    return df


def summarize_labels(df: pd.DataFrame) -> None:
    print("risk_factor_count distribution (0 = clean, 3 = all three threat axes active):")
    print(df["risk_factor_count"].value_counts().sort_index())
    print(f"\nAny risk factor present: {(df['risk_factor_count'] > 0).mean():.4f}")
    for axis in ["gt_cpo_malicious", "gt_emsp_malicious", "gt_ev_sybil"]:
        print(f"  {axis} rate: {df[axis].mean():.4f}")
    print(f"\ncomposite_risk_score describe:\n{df['composite_risk_score'].describe()}")


if __name__ == "__main__":
    import os
    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = add_ground_truth_labels(df)
    summarize_labels(df)


In [ ]:
%%writefile temporal_split.py
"""
temporal_split.py
--------------------
Train/test split for baseline evaluation: temporal within each world,
not random and not by-world.

WHY TEMPORAL: day_idx is REAL (derived from actual ACN-Data connection
timestamps, see raw_preprocess.py) and IDENTICAL in range across both
worlds -- only the synthetic identity/fraud fields differ by world, not
the real session timing. injection.py's attacks are keyed on day_idx
(onset_day), so a model that has only ever seen days before the cutoff
has a genuinely harder, more realistic job on the test split than a
model trained on a random shuffle that leaks future-day sessions into
training. This is the right split for a benchmark meant to measure "how
well/fast does each baseline detect or adapt to a trust problem," not
just "can it memorize this distribution."

Applied with the SAME day cutoff across all worlds (computed once, on
the full combined frame), since day_idx's distribution is shared real
data, not an independent per-world draw.

ONE DISCLOSED CAVEAT: enriched_features.add_baseline_charge_duration's
station+hour empirical-median fallback was computed over the WHOLE
dataset (train+test combined) when the benchmark CSV was generated, not
separately per split -- a minor leakage source for that one contextual
feature column specifically. Harmless for anything that doesn't use
baseline_charge_duration_h as a predictor of risk (it isn't one), but
worth stating plainly rather than silently ignoring, consistent with
this project's disclosure discipline.
"""

import numpy as np
import pandas as pd

TRAIN_FRACTION_OF_DAYS = 0.7  # documented split point, not tuned


def compute_day_cutoff(df: pd.DataFrame, train_fraction: float = TRAIN_FRACTION_OF_DAYS) -> int:
    days = np.sort(df["day_idx"].unique())
    cutoff_idx = int(len(days) * train_fraction)
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both splits non-empty
    return int(days[cutoff_idx])


def temporal_train_test_split(df: pd.DataFrame, day_cutoff: int = None):
    if day_cutoff is None:
        day_cutoff = compute_day_cutoff(df)
    train = df[df["day_idx"] < day_cutoff].copy()
    test = df[df["day_idx"] >= day_cutoff].copy()
    print(f"[split] day_cutoff={day_cutoff}  train={len(train)} rows "
          f"(days {train['day_idx'].min()}-{train['day_idx'].max()})  "
          f"test={len(test)} rows (days {test['day_idx'].min()}-{test['day_idx'].max()})")
    for name, part in [("train", train), ("test", test)]:
        rate = (part["risk_factor_count"] > 0).mean() if "risk_factor_count" in part.columns else float("nan")
        print(f"  {name}: any-risk-factor rate = {rate:.4f}")
    return train, test, day_cutoff


if __name__ == "__main__":
    import os
    import benchmark_labels

    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = benchmark_labels.add_ground_truth_labels(df)
    train, test, cutoff = temporal_train_test_split(df)
    print(f"\nWorld breakdown -- train:\n{train['world_id'].value_counts()}")
    print(f"World breakdown -- test:\n{test['world_id'].value_counts()}")


In [ ]:
%%writefile ml_features.py
"""
ml_features.py
-----------------
Feature set and target shared by the ML and DL baselines. Every input
here is something a live system would actually have at scoring time --
nothing from another baseline's output (RBAC/ABAC decisions, naive
trust scores) and nothing derived from the ground-truth labels
themselves.

TARGET: binary any_risk = (risk_factor_count > 0) -- the same "any real
threat axis active" definition RBAC/ABAC are already judged against
(rbac_baseline.summarize_rbac_blindness, abac_baseline.summarize_single_vs_multi).
Each model's predicted P(any_risk=0) becomes its continuous [0,1] trust
score, fed through decision_engine.trust_weighted_decision() for the
3-way PERMIT/CONDITIONAL-PERMIT/DENY decision -- same as every other
baseline, not reduced to binary.

FEATURES:
  - billing_deviation, latency_deviation: the two fuzzy-engine inputs
    (real-grounded on the metered side, see feature_engineering.py) --
    legitimate raw signals, not another baseline's processed output.
  - certificate_valid, is_roaming, is_peak_hour: real session attributes.
  - previous_flag_rate: ALREADY a real column (enriched_features.py) --
    a causal, prior-only running eMSP dispute rate. Using it here is
    feature engineering, not reusing ABAC's naive_emsp_trust (which is
    just 1 - this same column, renamed for ABAC's own purposes).
  - baseline_charge_duration_h, energy_rate_anomaly: real engineered
    columns from enriched_features.py.
  - charge_duration_h, full_duration_h, idle_h, avg_charge_rate_kw:
    raw session telemetry.
  - ev_session_share_so_far: NEW here -- a causal running statistic (same
    spirit as naive_trust.py's EV computation, RE-DERIVED independently
    so this module has no coupling to the ABAC baseline's internals) --
    the fraction of all sessions seen so far (across the FULL
    chronological sequence, not per-split -- see add_ev_session_share_feature's
    docstring) that belong to this EV.
  - day_idx: raw numeric context.

NOT included, deliberately: ev_id itself (1000-way high-cardinality
identity -- the EV-share feature above is the generalizable signal; the
raw ID would let the model memorize specific train-period EV identities
rather than learn the actual reuse-pattern signature), every
RBAC/ABAC/naive-trust output column (would be benchmarking one baseline
against a feature built from another baseline's output, not an
independent comparison) -- AND, as of this revision, cpo_id/emsp_id too.

BUG LOG (kept here deliberately, not swept away -- same discipline as
fuzzy_trust.py's own bug log): the first version of this module DID
include cpo_id/emsp_id as one-hot categorical features. On the real
125,910-row benchmark this produced AUROC/AUPRC = 1.0000 EXACTLY for
both logistic regression and random forest -- not a good score, a tell.
Root cause: in this dataset the malicious CPO/eMSP are malicious from
day 0 with no time variation (onset_day=0), so cpo_id=="CPO-B" implies
gt_cpo_malicious=1 with zero exceptions, same for emsp_id=="eMSP-C" --
a model with those IDs as input doesn't need billing_deviation or
previous_flag_rate at all; it just memorizes "if name is CPO-B or
eMSP-C, flag it," which is a lookup table, not a behavioral detector,
and wouldn't generalize to a deployment with different CPO/eMSP names.
(A synthetic smoke test run before the real data was available had the
same confound baked into its own generator, which is why it showed a
suspiciously-high-but-not-quite-1.0 AUROC~0.995 instead of flagging the
problem outright -- the real data's exact 1.0000 was the actual tell.)
Fixed by dropping cpo_id/emsp_id entirely: every remaining feature is a
genuine behavioral/contextual signal (billing_deviation,
previous_flag_rate, ev_session_share_so_far, etc.), so whatever
separability the model achieves now reflects learning from behavior,
not memorizing which specific name is the known bad one -- the same
standard TrustMesh itself is held to (its trust score comes from
billing_deviation observations, not from knowing a name in advance).

SECOND BUG LOG ENTRY (found while wiring up the 7-build variant suite,
confirmed PRE-EXISTING -- present even for the original single-build,
2-world "main" dataset): add_ev_session_share_feature's running count
and its denominator (total_seen) were kept across the ENTIRE passed-in
frame with no awareness of which world or variant a row came from --
the same class of cross-world leak documented in naive_trust.py's own
BUG LOG (world 0 and world 1, and now all 7 variants, share the same
real connection_time values, so sorting the union together interleaves
independent synthetic universes instead of treating each as its own
clean timeline). Fixed the same way: grouped by (variant, world_id)
before computing the running share, with a no-grouping fallback when
those columns aren't present."""

import numpy as np
import pandas as pd

NUMERIC_FEATURES = [
    "billing_deviation", "latency_deviation", "previous_flag_rate",
    "baseline_charge_duration_h", "energy_rate_anomaly",
    "charge_duration_h", "full_duration_h", "idle_h", "avg_charge_rate_kw",
    "ev_session_share_so_far", "day_idx",
]
BOOLEAN_FEATURES = ["certificate_valid", "is_roaming", "is_peak_hour"]
ALL_FEATURES = NUMERIC_FEATURES + BOOLEAN_FEATURES
TARGET = "any_risk"


DEFAULT_GROUP_COLS = ("variant", "world_id")  # see SECOND BUG LOG entry above


def add_ev_session_share_feature(df: pd.DataFrame, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """Causal running per-EV session share. MUST be called on the FULL
    chronological sequence (train+test combined) BEFORE any train/test
    split -- calling it separately on an already-split test frame would
    reset every EV's running count to zero at the split boundary,
    understating exactly the reuse signature this feature exists to
    capture. (Independent of naive_trust.py's analogous computation,
    deliberately re-derived here so the ML feature set has no coupling
    to the ABAC baseline's internals.)

    Computed independently within each (variant, world_id) group when
    those columns are present (falls back to the whole frame as one
    group otherwise) -- see this module's SECOND BUG LOG entry."""
    df = df.copy()
    present = [c for c in group_cols if c in df.columns]
    groups = df.groupby(present, sort=False) if present else [(None, df)]

    share_col = pd.Series(np.nan, index=df.index, dtype=float)
    for _, g in groups:
        order = g.sort_values("connection_time").index
        ev = g.loc[order, "ev_id"].to_numpy()

        running_n: dict = {}
        total_seen = 0
        local = np.empty(len(g))
        for i, e in enumerate(ev):
            this_ev_n = running_n.get(e, 0)
            local[i] = (this_ev_n / total_seen) if total_seen > 0 else 0.0
            running_n[e] = this_ev_n + 1
            total_seen += 1
        share_col.loc[order] = local

    df["ev_session_share_so_far"] = share_col
    return df


def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TARGET] = (df["risk_factor_count"] > 0).astype(int)
    return df


def prepare_ml_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Adds the EV-share feature and target, and coerces the boolean
    columns to int (sklearn-friendly). Call on the FULL combined
    chronological sequence, before splitting -- see
    add_ev_session_share_feature's docstring."""
    df = add_ev_session_share_feature(df)
    df = add_target(df)
    for col in BOOLEAN_FEATURES:
        df[col] = df[col].astype(int)
    return df


## Step 3: clear stale checkpoints, then write and train the graph-structural baseline

Same unconditional-clear discipline as `05`/`07`/`08`'s own Step 2/Step 3 -- the output folder is wiped before training so a re-run of this notebook never silently reuses a checkpoint from a previous architecture change. `graph_baseline.py`'s own docstring has the full design rationale (the 2-hop aggregation scheme, why no learned entity embeddings, the causality rules, the full-batch-training rationale).


In [ ]:
import os
import shutil, os
graph_out_dir = os.environ.get('V2G_GRAPH_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/graph_baseline')
if os.path.isdir(graph_out_dir):
    shutil.rmtree(graph_out_dir)
    print(f'Cleared {graph_out_dir}')
else:
    print('Nothing to clear.')


In [ ]:
%%writefile graph_baseline.py
"""
graph_baseline.py
---------------------
Graph-structural baseline: a small 2-hop heterogeneous mean-aggregation
GNN ("GraphSAGE-lite") over the CPO-eMSP-EV relationship graph induced
by sessions, completing the baseline ladder's last non-TrustMesh rung
(RBAC -> ABAC -> ML -> DL -> RL -> **graph-structural** -> TrustMesh,
per `benchmark_labels.py`'s own docstring list).

WHAT MAKES THIS "GRAPH-STRUCTURAL" AND NOT JUST MORE FEATURE
ENGINEERING: every earlier baseline scores ONE session row using only
that row's OWN features. This one additionally lets a session's score
depend on an aggregate of EVERY OTHER session sharing its CPO, its
eMSP, or its EV (hop 1), AND on an aggregate of what THOSE other
entities' own session neighborhoods look like one level further out
(hop 2) -- i.e. actual multi-hop message passing over the
session-entity bipartite graph, implemented in plain PyTorch via
differentiable scatter-mean (no external graph library, so nothing here
depends on a torch_geometric install succeeding in a given Colab
runtime). Concretely:

  hop 0 (entity init):  mean of (scaled) ALL_FEATURES over an entity's
                         own incident sessions, plus two log-degree
                         scalars (how many distinct entities of each OTHER
                         type it has co-occurred with) -- NOT identity,
                         NOT a learned lookup table (see BUG-LOG-STYLE
                         NOTE below for why that distinction matters).
  hop 1 (session):      combine a session's own features with its 3
                         neighboring entities' hop-0 vectors (a learned
                         linear + ReLU layer).
  hop 2 (entity, then
         session again): re-aggregate hop-1 SESSION embeddings back onto
                         each entity (mean over its incident sessions),
                         then combine a session's hop-1 embedding with
                         its 3 neighbors' hop-1 entity embeddings (a
                         second learned linear + ReLU layer) -- this is
                         what lets an EV's final representation be
                         influenced by the aggregate behavior of every
                         OTHER entity its CPO/eMSP neighbors have also
                         dealt with, not just its own history.

BUG-LOG-STYLE NOTE (same discipline as ml_features.py's own BUG LOG --
read before changing how entities are represented here): entity nodes
are deliberately given NO learned, per-identity embedding (no
nn.Embedding keyed by cpo_id/emsp_id/ev_id). ml_features.py's BUG LOG
already documents why a model that can see WHICH name is malicious
reduces to a lookup table (AUROC/AUPRC = 1.0000 exactly) rather than a
behavioral detector that would generalize to a deployment with
different names. A learned entity-embedding table keyed by name is
exactly that lookup table with extra steps, and it would ALSO silently
break zero-shot cross-variant evaluation the same way (each variant's
embedding table would need retraining, defeating the point of the
train-on-'main'-only protocol). Every entity representation here is
therefore a DETERMINISTIC, non-learned function of neighboring sessions'
behavioral features -- generalizable signal, not a name lookup -- exactly
the same standard ml_features.py and TrustMesh itself are held to.

SECOND BUG LOG ENTRY: the note above rules out a LEARNED per-identity
lookup, but misses a second, subtler channel to the same effect. The
first version's hop-0 aggregate averaged a feature_cols mean over an
entity's ENTIRE context-split history. With N_CPO=N_EMSP=3
(identity_assignment.py) and tens of thousands of sessions per entity in
the training context, that average converges to a near-zero-variance
estimate of the entity's TRUE underlying per-entity rate parameter --
which, for the eMSP axis specifically, IS the injected signal itself
(feature_engineering.py's emsp_dispute_flag is a per-session Bernoulli
draw at a rate that is a clean, constant function of emsp_id; a single
session's previous_flag_rate carries real sampling noise around that
rate, which is exactly why ml_baseline.py/dl_baseline.py/rl_baseline.py
only reach 0.91-0.93 AUROC on this axis -- but averaging that noisy
per-session feature over thousands of the SAME 3 entities' sessions
removes nearly all of that noise, so the aggregate recovers the rate
parameter almost exactly). A confirmatory ablation
(09_graph_structural_baseline.ipynb, Step 6) scored the trained model
with each scored session's OWN features zeroed out, leaving only the
entity aggregates: the eMSP-axis AUROC barely moved (delta <= 0.0004
on every variant), confirming identity-via-over-aggregation, not
session-level behavior, was carrying that axis. The CPO axis was NOT
similarly inflated (same N=3, but gt_cpo_malicious additionally requires
day_idx >= onset_day, which a history-spanning average can't resolve);
the EV axis (N_SYNTHETIC_EVS=1000) only shows a mild, legitimate
denoising benefit from aggregation, not this effect.

FIX: MAX_NEIGHBOR_HISTORY bounds each entity's hop-0 feature-mean to at
most that many of its MOST RECENT context-split rows, instead of its
full history -- this is not an ad hoc patch but a restoration of
GraphSAGE's own standard discipline (Hamilton et al., 2017 sample a
fixed number of neighbors per hop for exactly this reason: an
unbounded-neighborhood aggregate trivially overfits/converges for any
node type with few, stable neighbors). This "lite" implementation had
dropped that sampling step for simplicity at this benchmark's small
scale, which is what let the no-raw-identity guarantee above be true in
letter but not in effect for N=3 axes. Entity DEGREE (how many distinct
OTHER-type entities this one has co-occurred with) is deliberately left
on the FULL context split, uncapped -- it is a count, not a per-entity
rate, and is not implicated in the mechanism above.

DOSE-RESPONSE, FIRST PASS (MAX_NEIGHBOR_HISTORY=25, Hamilton et al.'s own
S1 value): re-running the Step 6 ablation against this bound showed real
progress -- the eMSP-axis ablation delta moved from ~0 (pure identity)
to a consistent +0.006 to +0.025 (the session's own features now
measurably matter) -- but the real eMSP-axis AUROC stayed elevated at
0.88-0.997, still well above the ~0.91-0.93 every non-graph baseline
gets on the same axis. The arithmetic reason: emsp_dispute_flag's
Bernoulli rate gap between the malicious eMSP and the other two is large
enough that even 25 samples' standard error is small relative to it, so
25-row bounding reduces but does not eliminate the effect.

DOSE-RESPONSE, SECOND PASS (MAX_NEIGHBOR_HISTORY=5): tightening further
did NOT cleanly close the remaining gap -- it introduced a DIFFERENT
failure mode instead. Seed-reproducibility (seed_101/103/104 AUROC std,
stable at 0.0001-0.0003 for every baseline in this project including
this one at K=25) jumped to 0.0064, and 'main's own test AUROC dropped
to 0.9131 (massive false-positive spike) while seed_101/104 stayed
near-perfect (0.999, 0.9996) -- a swing between "nearly broken" and
"nearly perfect" across nominally-identical reseeds. Cause: at K=5, each
entity's hop-0 vector is a snapshot of just 5 rows, computed once from
'main's sub_train at TRAINING time and recomputed from a DIFFERENT
5-row sample (each variant's own full train split) at EVALUATION time
(see the CAUSALITY section below for why train/eval use different
context splits at all) -- a sample that tiny lets those two snapshots
differ enough by chance that the model overfits to training's arbitrary
5-row pattern rather than learning anything that transfers. Conclusion:
there is a floor below which bounding helps (25) and a point below
which it breaks (5); hunting for a sweet spot between them trades one
magic number for another with no independent justification. Reverted to
MAX_NEIGHBOR_HISTORY=25 below.

TARGETED FIX ON TOP OF K=25: ENTITY_EXCLUDED_FEATURES drops
previous_flag_rate from the eMSP entity's OWN hop-0 aggregate
specifically (every session's own copy of it, and its presence in the
CPO/EV entities' aggregates, are untouched) -- feature_engineering.py
constructs emsp_dispute_flag as a per-session Bernoulli draw at a rate
that is a clean, constant function of emsp_id, so previous_flag_rate
(its running average) is exactly the feature a bounded-but-nonzero
sample size still partially recovers (see the first dose-response entry
above). CPO's and EV's own axes never showed this inflation signature
in any of the ablations above, so neither gets an equivalent exclusion --
this is a scoped, evidence-driven removal of the one confirmed
tautological channel, not a blanket "drop whatever correlates with risk"
rule, which would gut the baseline's genuine behavioral signal elsewhere.

CAUSALITY / WHICH ROWS CONTRIBUTE TO AN ENTITY'S AGGREGATE -- the single
most important correctness property of this module, so stated plainly
rather than left implicit: at every hop, an entity's aggregate is built
ONLY from a designated CONTEXT split's rows, and a row being SCORED never
contributes to its own entity's aggregate.
  - During TRAINING: the context split is 'main''s sub_train portion
    (TRAIN_VARIANT's train days, minus a tail-of-days validation slice --
    same VAL_FRACTION_OF_TRAIN_DAYS carve-out as dl_baseline.py). Model
    weights are gradient-updated ONLY against sub_train rows; val rows
    are scored using sub_train-built entity aggregates and never update
    any weight (same held-out discipline as dl_baseline.py's early
    stopping).
  - During EVALUATION of variant X's test split (zero-shot, frozen
    weights): the context split is variant X's OWN FULL train split
    (day_idx < day_cutoff). ONE DISCLOSED CAVEAT, same disclosure
    discipline as temporal_split.py's own caveat: this means entity
    aggregates at evaluation time are built from MORE rows (the full
    train split) than the sub_train slice the weights were actually
    gradient-trained against -- deliberate, mirroring
    ml_features.add_ev_session_share_feature's "use the fullest
    available prior history" convention, and harmless for correctness
    since it is still strictly confined to that variant's OWN pre-cutoff
    train rows, never the test rows being scored.
  - An entity value present in a scored split but absent from its
    context split (e.g. a never-before-seen id) falls back to a
    dedicated "unknown" row: the context split's global mean features
    and zero degree -- a conservative, uninformative default, not an
    error.

SAME SCAFFOLDING AS EVERY OTHER BASELINE: ml_features.ALL_FEATURES,
train-once-on-'main'/zero-shot-eval-on-every-variant, the
world_id-keyed predictions-CSV join-key convention (see
ml_baseline.py's own comment for the bug this avoids), GPU-aware device
selection, checkpointing (.pt + scaler .joblib).

FULL-BATCH TRAINING, NOT MINI-BATCHED: hop 2's entity aggregate is a
function of EVERY sub_train row's own hop-1 embedding under the CURRENT
weights, so it must be recomputed over the WHOLE sub_train split on
every forward pass -- mini-batching would either be wrong (stale/partial
aggregates) or need far more machinery than this baseline warrants.
Full-batch gradient descent over the whole sub_train split each epoch is
the standard, expected way to train a graph model at this scale (the
same convention used for e.g. Cora/Citeseer-scale GNN benchmarks), and
is cheap here regardless -- a two-layer, ~60-66-dim-hidden network over
a few thousand sub_train rows, on GPU or CPU alike.
"""

import os
import sys
import time

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, os.path.dirname(__file__) or ".")

import ml_features
import benchmark_labels
import temporal_split
from decision_engine import trust_weighted_decision

OUT_DIR = os.environ.get(
    "V2G_GRAPH_OUT_DIR",
    "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/graph_baseline",
)

TRAIN_VARIANT = "main"
MODEL_NAME = "graphsage_lite"
SEED = 42

VAL_FRACTION_OF_TRAIN_DAYS = 0.15  # carved from the end of main's train days,
                                     # same temporal-not-random discipline as dl_baseline.py
HIDDEN1 = 64   # hop-1 session embedding dim -- same capacity as dl_baseline.MLP's first layer
HIDDEN2 = 32   # hop-2 session embedding dim -- same capacity as dl_baseline.MLP's second layer
LR = 1e-3
MAX_EPOCHS = 300   # full-batch GD needs more epochs than mini-batch SGD (one update/epoch, not many)
PATIENCE = 25       # early stop if val AUROC hasn't improved for this many epochs

ENTITY_COLS = {"cpo": "cpo_id", "emsp": "emsp_id", "ev": "ev_id"}
OTHER_TYPES = {"cpo": ["emsp", "ev"], "emsp": ["cpo", "ev"], "ev": ["cpo", "emsp"]}
DEGREE_DIM = 2  # one log-degree scalar per "other" entity type
ENTITY_DIM = len(ml_features.ALL_FEATURES) + DEGREE_DIM

MAX_NEIGHBOR_HISTORY = 25  # Hamilton et al.'s (2017) own S1 neighbor-sample value. 5 was tried
                             # and destabilized seed-reproducibility instead of closing the
                             # remaining gap further -- see the module docstring's
                             # "DOSE-RESPONSE, SECOND PASS" note. Reverted to 25, paired below
                             # with a targeted exclusion for the one axis still affected.

ENTITY_EXCLUDED_FEATURES = {
    "emsp": ["previous_flag_rate"],  # see module docstring's "TARGETED FIX ON TOP OF K=25"
}


def _device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SessionGNN(nn.Module):
    """Two learned layers: hop-1 combines a session's own features with its
    3 neighboring entities' hop-0 vectors; hop-2 combines the hop-1 session
    embedding with its 3 neighbors' hop-1 entity embeddings. See module
    docstring for what builds the entity vectors fed into each layer."""

    def __init__(self, session_dim, entity_dim=ENTITY_DIM, hidden1=HIDDEN1, hidden2=HIDDEN2):
        super().__init__()
        self.layer1 = nn.Linear(session_dim + 3 * entity_dim, hidden1)
        self.layer2 = nn.Linear(hidden1 + 3 * hidden1, hidden2)
        self.head = nn.Linear(hidden2, 1)

    def session_layer1(self, x_session, h_cpo0, h_emsp0, h_ev0):
        return torch.relu(self.layer1(torch.cat([x_session, h_cpo0, h_emsp0, h_ev0], dim=1)))

    def session_layer2_logits(self, h1, h_cpo1, h_emsp1, h_ev1):
        h2 = torch.relu(self.layer2(torch.cat([h1, h_cpo1, h_emsp1, h_ev1], dim=1)))
        return self.head(h2).squeeze(-1)


def model_checkpoint_path():
    return os.path.join(OUT_DIR, f"{MODEL_NAME}.pt")


def scaler_checkpoint_path():
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_scaler.joblib")


def variant_preds_path(variant_name):
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_{variant_name}_test_predictions.csv")


def _prepare_full_frame(combined_csv_path):
    """Identical to ml_baseline.py/dl_baseline.py/rl_baseline.py's
    _prepare_full_frame -- same frame, same feature computation -- so this
    baseline is judged on EXACTLY the same data as every other one."""
    df = pd.read_csv(combined_csv_path)
    df["connection_time"] = pd.to_datetime(df["connection_time"], utc=True)
    df = benchmark_labels.add_ground_truth_labels(df)
    df = ml_features.prepare_ml_frame(df)
    day_cutoff = temporal_split.compute_day_cutoff(df)
    return df, day_cutoff


def _train_val_day_cutoff(train_df, val_fraction=VAL_FRACTION_OF_TRAIN_DAYS):
    days = np.sort(train_df["day_idx"].unique())
    cutoff_idx = int(len(days) * (1 - val_fraction))
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both non-empty
    return int(days[cutoff_idx])


def scatter_mean(src, index, n_groups, device):
    """src: [N, D] (differentiable). index: LongTensor [N] in [0, n_groups-1].
    Returns [n_groups, D], the per-group mean of src -- groups with zero
    contributing rows return an all-zeros vector (clamped count=1)."""
    sums = torch.zeros(n_groups, src.shape[1], device=device, dtype=src.dtype).index_add(0, index, src)
    counts = torch.zeros(n_groups, device=device, dtype=src.dtype).index_add(
        0, index, torch.ones(src.shape[0], device=device, dtype=src.dtype)
    )
    counts = counts.clamp(min=1.0).unsqueeze(1)
    return sums / counts


def _build_entity_maps(context_df):
    """{type_key: {raw_id_value: int_index}} covering every entity VALUE
    seen in context_df. Index len(map) (one past the last real index) is
    reserved for the "unknown entity" fallback row -- see module docstring.
    Keys are kept at the column's OWN dtype (no str-casting) deliberately,
    so they line up exactly with groupby's index in _entity0_features below
    -- a str-cast key here would silently never match an int-typed groupby
    index and make every lookup fall back to "unknown"."""
    maps = {}
    for key, col in ENTITY_COLS.items():
        uniques = sorted(context_df[col].unique(), key=str)
        maps[key] = {v: i for i, v in enumerate(uniques)}
    return maps


def _entity_indices(rows_df, entity_maps):
    """{type_key: np.int64 array[len(rows_df)]} -- index into that type's
    entity0/entity1 tensor for each row, falling back to the "unknown" slot
    (index == len(that type's map)) for any value not in entity_maps."""
    out = {}
    for key, col in ENTITY_COLS.items():
        m = entity_maps[key]
        unknown = len(m)
        out[key] = rows_df[col].map(m).fillna(unknown).astype(np.int64).to_numpy()
    return out


def _entity0_features(context_df, entity_maps, max_history=MAX_NEIGHBOR_HISTORY):
    """{type_key: np.float32 array[n_type+1, ENTITY_DIM]} -- hop-0 entity
    vectors: mean of (already-scaled) ALL_FEATURES over at most the
    `max_history` MOST RECENT (by day_idx) of the entity's own context_df
    rows -- NOT its full history (see module docstring's SECOND BUG LOG
    ENTRY for why bounding this matters for this benchmark specifically) --
    plus log1p-degree against each OTHER entity type, computed on the
    entity's FULL (uncapped) context_df rows -- degree is a connectivity
    count, not a per-entity rate, and is not implicated in that bug log
    entry. ENTITY_EXCLUDED_FEATURES[key], if any, are left at the GLOBAL
    mean rather than this entity's own mean (see module docstring's
    "TARGETED FIX" note) -- every other feature aggregates normally. Row
    n_type (the unknown-entity fallback) gets the GLOBAL context_df mean
    features (uncapped) and zero degree. See module docstring's causality
    section: context_df here is ALWAYS the designated context split,
    never rows being scored."""
    feature_cols = ml_features.ALL_FEATURES
    feature_idx = {f: i for i, f in enumerate(feature_cols)}
    global_mean = context_df[feature_cols].mean().to_numpy(dtype=np.float32)

    # Sorted once, shared by every entity type's bounded-history slice below.
    recent_first = context_df.sort_values("day_idx", ascending=False)

    out = {}
    for key, col in ENTITY_COLS.items():
        m = entity_maps[key]
        n = len(m)
        arr = np.zeros((n + 1, ENTITY_DIM), dtype=np.float32)
        arr[:n, :len(feature_cols)] = global_mean  # default for any entity with an odd empty group
        arr[n, :len(feature_cols)] = global_mean    # unknown-entity fallback row

        full_grouped = context_df.groupby(col)
        other_cols = [ENTITY_COLS[o] for o in OTHER_TYPES[key]]
        degrees = full_grouped[other_cols].nunique()  # degree: full history, intentionally uncapped

        # Features this entity type is allowed to aggregate its OWN mean for -- excludes any
        # ENTITY_EXCLUDED_FEATURES[key] entry, which instead stays at the global_mean default
        # already written into arr above (no entity-specific information for that feature).
        excluded = set(ENTITY_EXCLUDED_FEATURES.get(key, []))
        agg_features = [f for f in feature_cols if f not in excluded]

        capped = recent_first.groupby(col, sort=False, group_keys=False).head(max_history)
        means = capped.groupby(col)[agg_features].mean()  # feature means: bounded-history only

        for raw_value, idx in m.items():
            if raw_value in means.index:
                row_means = means.loc[raw_value]
                for fname in agg_features:
                    arr[idx, feature_idx[fname]] = row_means[fname]
                deg_vals = degrees.loc[raw_value].to_numpy(dtype=np.float32)
                arr[idx, len(feature_cols):] = np.log1p(deg_vals)
            # else: stays at the global-mean / zero-degree default set above
        out[key] = arr
    return out


def _forward_logits(model, scored_X, scored_idx, entity0, ctx_X, ctx_idx, n_entities, device):
    """Logits for scored_X/scored_idx. ctx_X/ctx_idx/entity0/n_entities
    describe the designated CONTEXT split (sub_train during training, a
    variant's own full train split during evaluation -- see module
    docstring). Always recomputes hop-1 for both the context split and the
    scored rows from scratch (cheap at this data scale, and avoids any
    stale-cache bugs when scored rows ARE the context rows, e.g. during a
    training step)."""
    h_cpo0 = entity0["cpo"][scored_idx["cpo"]]
    h_emsp0 = entity0["emsp"][scored_idx["emsp"]]
    h_ev0 = entity0["ev"][scored_idx["ev"]]
    h1_scored = model.session_layer1(scored_X, h_cpo0, h_emsp0, h_ev0)

    ctx_h_cpo0 = entity0["cpo"][ctx_idx["cpo"]]
    ctx_h_emsp0 = entity0["emsp"][ctx_idx["emsp"]]
    ctx_h_ev0 = entity0["ev"][ctx_idx["ev"]]
    h1_ctx = model.session_layer1(ctx_X, ctx_h_cpo0, ctx_h_emsp0, ctx_h_ev0)

    entity1 = {
        key: scatter_mean(h1_ctx, ctx_idx[key], n_entities[key] + 1, device)
        for key in ENTITY_COLS
    }

    h_cpo1 = entity1["cpo"][scored_idx["cpo"]]
    h_emsp1 = entity1["emsp"][scored_idx["emsp"]]
    h_ev1 = entity1["ev"][scored_idx["ev"]]
    return model.session_layer2_logits(h1_scored, h_cpo1, h_emsp1, h_ev1)


def _to_device_tensors(idx_dict, device):
    return {k: torch.tensor(v, dtype=torch.long, device=device) for k, v in idx_dict.items()}


def get_or_train_model(df, day_cutoff):
    """Loads the checkpoint (model + scaler) if one already exists;
    otherwise trains on TRAIN_VARIANT's ("main") sub_train rows only
    (tail-of-days val slice held out for early stopping, never trained on)
    and saves both. Mirrors dl_baseline.py's get_or_train_model."""
    model_path = model_checkpoint_path()
    scaler_path = scaler_checkpoint_path()
    device = _device()
    session_dim = len(ml_features.ALL_FEATURES)
    model = SessionGNN(session_dim).to(device)

    if os.path.exists(model_path) and os.path.exists(scaler_path):
        print(f"[{MODEL_NAME}] checkpoint already exists -- loading, not retraining (device={device})")
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        scaler = joblib.load(scaler_path)
        return model, scaler, device

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    full_train_df = df[(df["variant"] == TRAIN_VARIANT) & (df["day_idx"] < day_cutoff)].copy()
    val_cutoff = _train_val_day_cutoff(full_train_df)
    sub_train_df = full_train_df[full_train_df["day_idx"] < val_cutoff].copy()
    val_df = full_train_df[full_train_df["day_idx"] >= val_cutoff].copy()

    scaler = StandardScaler().fit(sub_train_df[ml_features.ALL_FEATURES])
    sub_train_df[ml_features.ALL_FEATURES] = scaler.transform(sub_train_df[ml_features.ALL_FEATURES])
    val_df[ml_features.ALL_FEATURES] = scaler.transform(val_df[ml_features.ALL_FEATURES])

    entity_maps = _build_entity_maps(sub_train_df)
    n_entities = {k: len(m) for k, m in entity_maps.items()}
    entity0_np = _entity0_features(sub_train_df, entity_maps)
    entity0 = {k: torch.tensor(v, device=device) for k, v in entity0_np.items()}

    sub_train_idx = _to_device_tensors(_entity_indices(sub_train_df, entity_maps), device)
    val_idx = _to_device_tensors(_entity_indices(val_df, entity_maps), device)

    X_sub_train = torch.tensor(sub_train_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
    X_val = torch.tensor(val_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
    y_sub_train = sub_train_df[ml_features.TARGET].to_numpy().astype(np.float32)
    y_val = val_df[ml_features.TARGET].to_numpy()
    y_sub_train_t = torch.tensor(y_sub_train, device=device)

    n_pos = float(y_sub_train.sum())
    n_neg = float(len(y_sub_train) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_auroc = -1.0
    best_state = None
    epochs_without_improvement = 0
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits_train = _forward_logits(model, X_sub_train, sub_train_idx, entity0,
                                        X_sub_train, sub_train_idx, n_entities, device)
        loss = criterion(logits_train, y_sub_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_val = _forward_logits(model, X_val, val_idx, entity0,
                                          X_sub_train, sub_train_idx, n_entities, device)
            val_proba = torch.sigmoid(logits_val).cpu().numpy()
        val_auroc = roc_auc_score(y_val, val_proba)

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0 or epochs_without_improvement >= PATIENCE:
            print(f"  epoch {epoch:3d}  loss={loss.item():.4f}  val_auroc={val_auroc:.4f}  "
                  f"best={best_val_auroc:.4f}  no_improve={epochs_without_improvement}")

        if epochs_without_improvement >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (no val AUROC improvement for {PATIENCE} epochs)")
            break

    model.load_state_dict(best_state)
    model.eval()

    os.makedirs(OUT_DIR, exist_ok=True)
    torch.save(model.state_dict(), model_path)
    joblib.dump(scaler, scaler_path)
    print(f"[{MODEL_NAME}] trained on '{TRAIN_VARIANT}' only: n_sub_train={len(sub_train_df)}  "
          f"n_val={len(val_df)}  best_val_auroc={best_val_auroc:.4f}  "
          f"({time.time() - t0:.1f}s, device={device}) -> {model_path}")
    return model, scaler, device


def evaluate_on_variant(model, scaler, device, df, day_cutoff, variant_name):
    """Scores the ALREADY-TRAINED (on 'main' only) network against
    variant_name's OWN test split -- zero-shot, no retraining. The CONTEXT
    split used for this variant's entity aggregates is its OWN FULL train
    split (day_idx < day_cutoff), per the module docstring's disclosed
    caveat. Mirrors ml_baseline.py/dl_baseline.py's evaluate_on_variant,
    including why world_id is saved for the join key."""
    train_df = df[(df["variant"] == variant_name) & (df["day_idx"] < day_cutoff)].copy()
    test_df = df[(df["variant"] == variant_name) & (df["day_idx"] >= day_cutoff)].copy()

    train_df[ml_features.ALL_FEATURES] = scaler.transform(train_df[ml_features.ALL_FEATURES])
    test_df[ml_features.ALL_FEATURES] = scaler.transform(test_df[ml_features.ALL_FEATURES])

    entity_maps = _build_entity_maps(train_df)
    n_entities = {k: len(m) for k, m in entity_maps.items()}
    entity0_np = _entity0_features(train_df, entity_maps)
    entity0 = {k: torch.tensor(v, device=device) for k, v in entity0_np.items()}

    ctx_idx = _to_device_tensors(_entity_indices(train_df, entity_maps), device)
    test_idx = _to_device_tensors(_entity_indices(test_df, entity_maps), device)
    X_ctx = torch.tensor(train_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
    X_test = torch.tensor(test_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
    y_test = test_df[ml_features.TARGET].to_numpy()

    model.eval()
    with torch.no_grad():
        logits_test = _forward_logits(model, X_test, test_idx, entity0, X_ctx, ctx_idx, n_entities, device)
    proba_risk = torch.sigmoid(logits_test).cpu().numpy()
    trust_score = 1.0 - proba_risk

    auroc = roc_auc_score(y_test, proba_risk)
    auprc = average_precision_score(y_test, proba_risk)
    decisions = [trust_weighted_decision(1.0, float(t)).value for t in trust_score]

    preds_path = variant_preds_path(variant_name)
    out = test_df[["session_id", "variant", "world_id", "cpo_id", "emsp_id", "ev_id", "day_idx",
                    "risk_factor_count", "any_risk"]].copy()
    out["trust_score"] = trust_score
    out["decision"] = decisions
    out.to_csv(preds_path, index=False)

    return {"model": MODEL_NAME, "variant": variant_name,
            "auroc": float(auroc), "auprc": float(auprc), "n_test": int(len(test_df))}


def main(combined_csv_path):
    df, day_cutoff = _prepare_full_frame(combined_csv_path)
    variants = sorted(df["variant"].unique()) if "variant" in df.columns else [None]
    device = _device()
    print(f"day_cutoff={day_cutoff}  variants found: {variants}")
    print(f"Training 1 model ({MODEL_NAME}) on '{TRAIN_VARIANT}' only, then evaluating "
          f"zero-shot on every variant's own test split (device={device})")

    model, scaler, device = get_or_train_model(df, day_cutoff)

    all_rows = []
    for variant_name in variants:
        row = evaluate_on_variant(model, scaler, device, df, day_cutoff, variant_name)
        all_rows.append(row)
        print(f"  [{MODEL_NAME:16s} -> {variant_name:16s}] "
              f"AUROC={row['auroc']:.4f}  AUPRC={row['auprc']:.4f}  n_test={row['n_test']}")

    summary = pd.DataFrame(all_rows)
    summary_path = os.path.join(OUT_DIR, "graph_baseline_variant_summary.csv")
    os.makedirs(OUT_DIR, exist_ok=True)
    summary.to_csv(summary_path, index=False)

    print(f"\n=== Summary table (trained once on '{TRAIN_VARIANT}', zero-shot evaluated "
          f"per variant) -- saved to {summary_path} ===")
    print(summary.set_index("variant")[["auroc", "auprc"]].round(4).to_string())

    seed_variants = [v for v in variants if v.startswith("seed_")]
    if seed_variants:
        print(f"\n=== Stability check across {len(seed_variants)} independent reseeds "
              f"({', '.join(seed_variants)}) -- mean +/- std AUROC ===")
        stab = summary[summary["variant"].isin(seed_variants)]["auroc"]
        print(f"  mean={stab.mean():.4f}  std={stab.std():.4f}")

    return summary


if __name__ == "__main__":
    combined_csv_path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    main(combined_csv_path)


In [ ]:
import os
import graph_baseline

combined_path = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
summary = graph_baseline.main(combined_path)


## Step 4: insights -- AUROC per variant, and stability across seeds


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

GRAPH_OUT_DIR = os.environ.get('V2G_GRAPH_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/graph_baseline')
summary = pd.read_csv(os.path.join(GRAPH_OUT_DIR, 'graph_baseline_variant_summary.csv'))
summary = summary.set_index('variant')
print(summary[['auroc', 'auprc']].round(4))

ax = summary['auroc'].plot(kind='bar', figsize=(9, 5), ylim=(0, 1.05), legend=False, color='#5470c6')
ax.set_ylabel('AUROC (test split, zero-shot per variant)')
ax.set_title("Graph-structural baseline: trained once on 'main', evaluated zero-shot per variant")
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('graph_baseline_variant_insights.pdf', dpi=300)
plt.show()
print('Saved graph_baseline_variant_insights.pdf')

seed_rows = summary[summary.index.str.startswith('seed_')]
if len(seed_rows):
    print(f"\nStability across {len(seed_rows)} independent reseeds -- "
          f"mean={seed_rows['auroc'].mean():.4f}  std={seed_rows['auroc'].std():.4f}")


## Step 5: F1/precision/recall/confusion matrix, per-axis AUROC, duplicate-key sanity check

Same `(session_id, variant, world_id)` join-key discipline as `05`/`07`/`08`'s own Step 4/5 -- the assert below is load-bearing, not decorative. No `drop_duplicates` band-aid: the underlying duplicate-raw-row issue is fixed at the source in `raw_preprocess.py` (notebook `02`), so this assert is expected to pass cleanly on its own.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                              confusion_matrix, classification_report, roc_auc_score,
                              average_precision_score, brier_score_loss)
import benchmark_labels
import glob, os as _os

OUT_DIR = os.environ.get('V2G_GRAPH_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/graph_baseline')
COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
MODEL_NAME = 'graphsage_lite'
DECISION_THRESHOLD = 0.5
JOIN_KEYS = ['session_id', 'variant', 'world_id']

full_labels = pd.read_csv(COMBINED_CSV)
full_labels = benchmark_labels.add_ground_truth_labels(full_labels)

dup_count = full_labels.duplicated(subset=JOIN_KEYS).sum()
print(f'duplicate rows on {JOIN_KEYS}: {dup_count} (expect 0 -- see this cell\'s markdown)')
assert dup_count == 0, f'{dup_count} duplicate rows on {JOIN_KEYS} in the combined CSV -- stop and investigate'
labels_df = full_labels[JOIN_KEYS + ['gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']]

variants = sorted({
    _os.path.basename(p)[len(f'{MODEL_NAME}_'):-len('_test_predictions.csv')]
    for p in glob.glob(_os.path.join(OUT_DIR, f'{MODEL_NAME}_*_test_predictions.csv'))
})
print(f'variants found from saved prediction files: {variants}\n')

overall_rows, axis_rows = [], []
for variant in variants:
    preds = pd.read_csv(f'{OUT_DIR}/{MODEL_NAME}_{variant}_test_predictions.csv')
    assert 'world_id' in preds.columns, f'{variant}: predictions file has no world_id column'
    n_before = len(preds)
    preds = preds.merge(labels_df, on=JOIN_KEYS, how='left')
    assert len(preds) == n_before, f'{variant}: join fanned out ({n_before} -> {len(preds)})'

    y_true = preds['any_risk'].astype(int).to_numpy()
    proba_risk = 1.0 - preds['trust_score'].to_numpy()
    y_pred = (proba_risk >= DECISION_THRESHOLD).astype(int)
    flagged_by_decision = preds['decision'].isin(['DENY', 'CONDITIONAL-PERMIT']).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall_rows.append({
        'variant': variant,
        'auroc': roc_auc_score(y_true, proba_risk),
        'auprc': average_precision_score(y_true, proba_risk),
        'brier': brier_score_loss(y_true, proba_risk),
        'f1@0.5': f1_score(y_true, y_pred, zero_division=0),
        'precision@0.5': precision_score(y_true, y_pred, zero_division=0),
        'recall@0.5': recall_score(y_true, y_pred, zero_division=0),
        'accuracy@0.5': accuracy_score(y_true, y_pred),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'f1_decision': f1_score(y_true, flagged_by_decision, zero_division=0),
    })

    for axis in ['gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']:
        y_axis = preds[axis].astype(int).to_numpy()
        axis_auroc = float('nan') if (y_axis.sum() == 0 or y_axis.sum() == len(y_axis)) else roc_auc_score(y_axis, proba_risk)
        axis_rows.append({'variant': variant, 'axis': axis, 'auroc': axis_auroc})

overall = pd.DataFrame(overall_rows)
by_axis = pd.DataFrame(axis_rows)
metrics_path = f'{OUT_DIR}/graph_baseline_variant_metrics_full.csv'
overall.to_csv(metrics_path, index=False)
print(f'Saved -> {metrics_path}\n')

print('=== overall (any_risk) metrics ===')
print(overall.set_index('variant')[
    ['auroc', 'auprc', 'brier', 'f1@0.5', 'precision@0.5', 'recall@0.5', 'accuracy@0.5', 'f1_decision']
].round(4).to_string())
print()
print('=== confusion matrix (tn/fp/fn/tp @ proba>=0.5) ===')
print(overall.set_index('variant')[['tn', 'fp', 'fn', 'tp']].to_string())
print()
print('=== per-axis AUROC (does the SAME score separate each individual threat axis?) ===')
print(by_axis.pivot(index='variant', columns='axis', values='auroc').round(4).to_string())
print()

for variant in variants:
    preds = pd.read_csv(f'{OUT_DIR}/{MODEL_NAME}_{variant}_test_predictions.csv')
    y_true = preds['any_risk'].astype(int).to_numpy()
    proba_risk = 1.0 - preds['trust_score'].to_numpy()
    y_pred = (proba_risk >= DECISION_THRESHOLD).astype(int)
    print(f'--- {variant} ---')
    print(classification_report(y_true, y_pred, target_names=['no_risk', 'any_risk'], digits=4, zero_division=0))


## Step 6: confirmatory ablation -- does entity identity alone explain the near-perfect AUROC?

Step 5's per-axis table shows `gt_emsp_malicious` AUROC at 0.97-1.0 -- well above the 0.91-0.93 every other baseline gets on the SAME axis from the SAME raw features, with no entity aggregation. The suspected reason: `gt_emsp_malicious = (emsp_id == MALICIOUS_EMSP)` is a pure, untimed function of identity, there are only 3 distinct eMSPs, and this module's hop-0 entity vector (`groupby(emsp_id)[features].mean()`) is a deterministic BIJECTION of emsp_id at that cardinality -- i.e. informationally equivalent to a 3-way identity lookup table, regardless of there being no *learned* embedding. `gt_cpo_malicious` has the same 3-way cardinality but an additional `day_idx >= onset_day` gate the identity fingerprint can't resolve (hence its AUROC stays moderate); `gt_ev_sybil` is untimed like eMSP but spread over up to 1000 distinct EVs, diluting the same effect.

This cell tests that directly: re-scores the ALREADY-TRAINED checkpoint (loaded, not retrained) on every variant's test split with each scored session's OWN 14 features zeroed out, while leaving its CPO/eMSP/EV entity-aggregate vectors genuine (built from the real context split, exactly as `evaluate_on_variant` does). If the model's per-axis AUROC barely drops versus Step 5's real numbers -- especially on the eMSP axis -- that is direct, quantitative confirmation that identity alone (not the session's own behavior) is carrying the near-perfect score.


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import joblib
from sklearn.metrics import roc_auc_score

import graph_baseline as gb

COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')

model_path = gb.model_checkpoint_path()
scaler_path = gb.scaler_checkpoint_path()
assert os.path.exists(model_path) and os.path.exists(scaler_path), (
    "Missing checkpoint/scaler -- run Step 3's cells above first. This ablation LOADS the "
    "already-trained model; it does not train anything itself.")

df, day_cutoff = gb._prepare_full_frame(COMBINED_CSV)
device = gb._device()

model = gb.SessionGNN(len(gb.ml_features.ALL_FEATURES)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
scaler = joblib.load(scaler_path)

AXES = [('cpo', 'gt_cpo_malicious'), ('emsp', 'gt_emsp_malicious'), ('ev', 'gt_ev_sybil')]


def axis_auroc(y_axis, score):
    y_axis = np.asarray(y_axis)
    if y_axis.sum() == 0 or y_axis.sum() == len(y_axis):
        return float('nan')
    return roc_auc_score(y_axis, score)


def score_variant(variant_name, ablate):
    """Mirrors graph_baseline.evaluate_on_variant's context/scoring split exactly -- same
    train_df/test_df, same entity_maps/entity0/ctx_idx built from the real context split in
    BOTH cases. When ablate=True, only the SCORED (test) rows' own session features are
    replaced with zeros right before the forward pass; entity0/entity1 aggregates are still
    built from the real, un-ablated context (train) split, unchanged from the real evaluation.
    This isolates how much of the score comes from 'which CPO/eMSP/EV is this session's'
    (via the entity-aggregate lookup alone) versus the session's own behavior."""
    train_df = df[(df['variant'] == variant_name) & (df['day_idx'] < day_cutoff)].copy()
    test_df = df[(df['variant'] == variant_name) & (df['day_idx'] >= day_cutoff)].copy()

    train_df[gb.ml_features.ALL_FEATURES] = scaler.transform(train_df[gb.ml_features.ALL_FEATURES])

    entity_maps = gb._build_entity_maps(train_df)
    n_entities = {k: len(m) for k, m in entity_maps.items()}
    entity0_np = gb._entity0_features(train_df, entity_maps)
    entity0 = {k: torch.tensor(v, device=device) for k, v in entity0_np.items()}

    ctx_idx = gb._to_device_tensors(gb._entity_indices(train_df, entity_maps), device)
    test_idx = gb._to_device_tensors(gb._entity_indices(test_df, entity_maps), device)
    X_ctx = torch.tensor(train_df[gb.ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)

    if ablate:
        X_test = torch.zeros((len(test_df), len(gb.ml_features.ALL_FEATURES)),
                              dtype=torch.float32, device=device)
    else:
        test_scaled = test_df.copy()
        test_scaled[gb.ml_features.ALL_FEATURES] = scaler.transform(test_df[gb.ml_features.ALL_FEATURES])
        X_test = torch.tensor(test_scaled[gb.ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)

    with torch.no_grad():
        logits = gb._forward_logits(model, X_test, test_idx, entity0, X_ctx, ctx_idx, n_entities, device)
    proba_risk = torch.sigmoid(logits).cpu().numpy()

    row = {'variant': variant_name, 'n_test': len(test_df),
           'overall_auroc': axis_auroc(test_df[gb.ml_features.TARGET].to_numpy(), proba_risk)}
    for axis, gt_col in AXES:
        row[f'{axis}_axis_auroc'] = axis_auroc(test_df[gt_col].to_numpy(), proba_risk)
    return row


variants = sorted(df['variant'].unique())
real_rows = [score_variant(v, ablate=False) for v in variants]
ablated_rows = [score_variant(v, ablate=True) for v in variants]

real_df = pd.DataFrame(real_rows).set_index('variant')
ablated_df = pd.DataFrame(ablated_rows).set_index('variant')

print("=== REAL (session's own features + entity aggregates) -- recomputed fresh, should match Step 5 ===")
print(real_df.round(4).to_string())
print("\n=== ABLATED (session's own 14 features ZEROED, entity aggregates genuine) ===")
print(ablated_df.round(4).to_string())

delta = (ablated_df - real_df).add_suffix('_delta')
print("\n=== Delta (ablated - real) -- near zero on an axis means identity ALONE already explains it ===")
print(delta[[f'{axis}_axis_auroc_delta' for axis, _ in AXES] + ['overall_auroc_delta']].round(4).to_string())

summary_path = f'{gb.OUT_DIR}/graph_baseline_entity_only_ablation.csv'
real_df.add_prefix('real_').join(ablated_df.add_prefix('ablated_')).to_csv(summary_path)
print(f"\nSaved {summary_path}")

print("\nIf ablated_emsp_axis_auroc stays close to real_emsp_axis_auroc across variants, that is")
print("direct confirmation that the near-perfect eMSP-axis AUROC comes from the entity-aggregate")
print("identity lookup (only 3 distinct eMSPs, untimed ground truth), not from the model learning")
print("any real session-level eMSP-fraud behavior -- meaning this baseline's headline numbers are")
print("not yet safe to report uncorrected.")


## Done

Output: `graph_baseline_variant_summary.csv` (AUROC/AUPRC per variant) and `graph_baseline_variant_metrics_full.csv` (the fuller F1/precision/recall/confusion-matrix/per-axis table) in `.../worlds/graph_baseline/`, plus the trained network (`graphsage_lite.pt`) and its fitted scaler (`graphsage_lite_scaler.joblib`) so re-running this notebook doesn't retrain from scratch.

Paste back: the training log (epochs + early-stopping point, device), the per-variant AUROC/AUPRC table, the stability check across the 3 reseeds, and the Step 5 metrics table -- directly comparable to `05`/`07`/`08`'s own tables, same features/target/protocol, a structurally different model (the only baseline in the ladder whose score depends on more than one session row at a time).
